#Loading Prepared Data

In [ ]:
import os
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
notebook_path = '/content/drive/My Drive/Python Projects/Walmart Sales Forecast'
save_path = os.path.join(notebook_path, 'data')

In [ ]:
train_file = os.path.join(save_path, "train_merged.csv")

In [ ]:
train = pd.read_csv(train_file)

Check

In [ ]:
train.shape

(421570, 17)

In [ ]:
train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday_x,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday_y,Type,Size
0,1,1,2010-02-05,24924.50,0,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,0,0,151315
1,1,1,2010-02-12,46039.49,1,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,1,0,151315
2,1,1,2010-02-19,41595.55,0,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,0,0,151315
3,1,1,2010-02-26,19403.54,0,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,0,0,151315
4,1,1,2010-03-05,21827.90,0,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,0,0,151315


#Performing EDA

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.express.colors import sequential
import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde
import plotly.io as pio

####Distribution of Weekly Sales

In [ ]:
pastel_colors = ['#fa9c93', '#83c9f4', '#ade366']

In [ ]:
fig1 = px.histogram(train, x='Weekly_Sales', nbins=100,
                   marginal='box', color='Type',  color_discrete_sequence=pastel_colors,
                   title='Distribution of Weekly Sales by Store Type')

fig1.update_layout(
    title={
        'text': 'Distribution of Weekly Sales by Store Type',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    xaxis_title='Weekly Sales ($)',
    yaxis_title='Count'
)

div1 = pio.to_html(fig1, full_html=False, include_plotlyjs='cdn')
fig1.show()

####Insights:

*   Store Type 0:
  *   Moderate but variable sales.
  *   Fewer extreme outliers.

*   Store Type 1:
  *   Largest sales volumes, widest spread of weekly sales.
  *   High variability → demand shocks or seasonal surges.

*   Store Type 2:
  *   Small, consistent sales.
  *   Low variability → more stable performance, but limited contribution to total revenue.


---


=> Highly right-skewed

=> Outliers (Type 1) need handling (log-scaling, robust metrics).

=> Single model will not capture the heterogeneity; forecasting should be store-type specific.



---

Revenue is disproportionately driven by Type 1 stores (a few high-sales outliers carry much of the weight).

Type 2 stores, while stable, have limited impact on total sales forecasts but may be operationally important (predictable supply chain).


####Total Weekly Sales Over Time

In [ ]:
weekly_sales = train.groupby('Date')['Weekly_Sales'].sum().reset_index()

In [ ]:
fig2 = px.line(weekly_sales,
              x='Date', y='Weekly_Sales',
              color_discrete_sequence=['#006ba6'],
              title='Total Weekly Sales Over Time')

fig2.update_layout(
    title={
        'text': 'Total Weekly Sales Over Time',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    yaxis_title='Weekly Sales ($)'
)

div2 = pio.to_html(fig2, full_html=False, include_plotlyjs=False)
fig2.show()

####Insights:

*   Stable sales: \$42M–\$50M range

*   Seasonal spikes (Dec–Jan):  \$65M–\$80M

*   Sharp drops before holiday surges: Demand shifts (customers postpone purchases, then buy heavily during holidays)

*   Pattern Repitition: 2010, 2011, 2012 → indicates strong seasonal seasonality


---

=> Models need holiday indicators and seasonal adjustments.

=> Long-term growth is not evident; sales are driven more by events than organic growth.




####Holiday-Specific Sales Impact

In [ ]:
fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=weekly_sales['Date'],
    y=weekly_sales['Weekly_Sales'],
    mode='lines',
    name='Weekly Sales'
))

holidays = {
    'SuperBowl': pd.to_datetime(['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08']),
    'LaborDay': pd.to_datetime(['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06']),
    'Thanksgiving': pd.to_datetime(['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29']),
    'Christmas': pd.to_datetime(['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27'])
}

for holiday, dates in holidays.items():
    for date in dates:
        fig3.add_shape(
            type="line",
            x0=date, x1=date,
            y0=0, y1=weekly_sales['Weekly_Sales'].max(),
            line=dict(color="red", dash="dash"),
        )
        fig3.add_annotation(
            x=date,
            y=weekly_sales['Weekly_Sales'].max(),
            text=holiday,
            showarrow=True,
            arrowhead=2,
            yshift=10
        )

fig3.update_layout(
    title={
        'text': 'Total Weekly Sales with Holiday Highlights',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    xaxis_title="Date",
    yaxis_title="Weekly Sales ($)",
    margin=dict(l=80, r=40, t=80, b=120),
    width=4000,
    height=600
)

div3 = pio.to_html(fig3, full_html=False, include_plotlyjs=False)
fig3.show()

####Insights:

*   Holidays: Thanksgiving, Black Friday, and Christmas → holidays highest peaks each year

*   Super Bowl effect:

  *   Noticeable but smaller spikes compared to Thanksgiving/Christmas
  *   Increased demand: likely food, beverages, party items

*   Labor Day: Little to no visible impact on total weekly sales.

---


=> Business planning (inventory, staffing, supply chain) should prioritize Q4 holidays as they are the largest revenue contributors.

=> Non-peak weeks are more predictable and stable → better suited for baseline trend analysis.



####Effect of Holidays

In [ ]:
# Calculate average weekly sales for holiday vs non-holiday
holiday_sales = train.groupby('IsHoliday_x')['Weekly_Sales'].mean().reset_index()
holiday_sales['Holiday_Label'] = holiday_sales['IsHoliday_x'].map({0: 'Non-Holiday', 1: 'Holiday'})

In [ ]:
fig4 = px.bar(
    holiday_sales,
    x='Holiday_Label',
    y='Weekly_Sales',
    title='Effect of Holidays: Average Weekly Sales Comparison',
    color='Holiday_Label',
    color_discrete_map={'Non-Holiday': '#f57569', 'Holiday': '#a7e655'},
    hover_data={'IsHoliday_x': True, 'Weekly_Sales': ':,.0f'},
    text='Weekly_Sales'
)

fig4.update_layout(
    title={
        'text': 'Effect of Holidays: Average Weekly Sales Comparison',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    xaxis_title='Period Type',
    yaxis_title='Average Weekly Sales ($)',
    showlegend=False,
    plot_bgcolor='white',
    paper_bgcolor='white',
    hovermode='x unified',
    font={'size': 12},
    margin=dict(t=80, b=60, l=80, r=60),
    yaxis=dict(
        gridcolor='lightgray',
        gridwidth=1,
        zeroline=True,
        zerolinecolor='gray',
        zerolinewidth=1
    ),
    xaxis=dict(
        gridcolor='lightgray',
        gridwidth=1
    )
)

# Add annotations for insights
max_sales = holiday_sales['Weekly_Sales'].max()
min_sales = holiday_sales['Weekly_Sales'].min()
difference = max_sales - min_sales
percentage_increase = (difference / min_sales) * 100

# Add annotation to display the insights
fig4.add_annotation(
    text=f"Holiday sales are ${difference:,.0f} ({percentage_increase:.1f}%) higher than non-holiday",
    xref="paper", yref="paper",
    x=0.5, y=1.07,
    showarrow=False,
    bgcolor="#d7eef5"
)

# Add interactive features
fig4.update_traces(
    marker=dict(
        opacity=0.8
    )
)

div4 = pio.to_html(fig4, full_html=False, include_plotlyjs=False)
fig4.show()

####Insights:

*   Average weekly sales:

  *   Holidays = \$17,035
  *   Non-holidays = \$15,901.

*   Difference = $1,134 uplift (~7.1%).

*   Holiday weeks boost sales, but magnitude is modest compared to the huge spikes seen around Thanksgiving/Christmas.



####Weekly Sales Trend by Store Type

In [ ]:
pastel_colors = ['#fa877d', '#49b2f2', '#a2e34d']

In [ ]:
type_sales = train.groupby(['Date','Type'])['Weekly_Sales'].sum().reset_index()

In [ ]:
fig5 = px.line(type_sales,
              x='Date', y='Weekly_Sales',
              color='Type', color_discrete_sequence=pastel_colors,
              title='Weekly Sales Trend by Store Type')

fig5.update_layout(
    title={
        'text': 'Weekly Sales Trend by Store Type',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    yaxis_title='Weekly Sales ($)'
)

div5 = pio.to_html(fig5, full_html=False, include_plotlyjs=False)
fig5.show()

####Insights:

*   Store Type 0 (A):

  *   Consistently the highest revenue contributor → \$30M  weekly.
  *   Revenue growth is dominated; strategic investment here gives the largest payoff.


*   Store Type 1 (B):

  *   Mid-level stores → ~\$12M–\$15M weekly.
  *   Contribute steadily, less volatile, useful for stable revenue streams.


*   Store Type 2 (C):

  *   Very small baseline → ~\$2M–\$3M weekly.
  *   Minimal holiday uplift → stable but marginal impact.
  *   Likely Neighborhood/Express stores, important for local presence but not major revenue drivers.



####Average Sales by Store

In [ ]:
store_sales = train.groupby('Store')['Weekly_Sales'].mean().sort_values(ascending=False).head(20).reset_index()

In [ ]:
fig6 = px.bar(store_sales,
             x='Store', y='Weekly_Sales',
             color='Store', color_continuous_scale='viridis',
             title='Top 20 Stores by Average Sales')

fig6.update_layout(
    title={
        'text': 'Top 20 Stores by Average Sales',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    yaxis_title='Weekly Sales ($)'
)

div6 = pio.to_html(fig6, full_html=False, include_plotlyjs=False)
fig6.show()

####Insights:

*   Top-performing stores average: ~\$28k–\$30k

*   Lower end of the top 20 sits around: \$15k–\$18k

*   Clear performance gap: power-law style contribution (a few stores dominate)

*   Store geography and customer demographics are critical predictors, not just store type.




---


*   Inventory allocation (ensure no stockouts in peak weeks).
*   Marketing campaigns (localized promotions yield bigger returns).
*   Staffing optimization (extra workforce during holiday surges).




####Average Sales by Department

In [ ]:
dept_sales = train.groupby('Dept')['Weekly_Sales'].mean().sort_values(ascending=False).head(20).reset_index()

In [ ]:
fig7 = px.bar(dept_sales,
             x='Dept', y='Weekly_Sales',
             color='Dept', color_continuous_scale='viridis',
             title='Top 20 Departments by Average Sales')

fig7.update_layout(
    title={
        'text': 'Top 20 Departments by Average Sales',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    yaxis_title='Weekly Sales ($)'
)

div7 = pio.to_html(fig7, full_html=False, include_plotlyjs=False)
fig7.show()

####Insights:

*   Top-performing departments: \$40k - \$75k

*   ~Dept 38, 65, 72, 92–95: dominate sales → highly skewed distribution (Pareto effect)


---

=> Core revenue-driving departments: groceries, household, pharmacy; prioritized in supply chain planning, promotions, and shelf space allocation.

=> Departments with consistently lower averages: customer basket diversity (cross-sell potential)


####Numerical Feature Distributions

In [ ]:
numerical_features = ['Temperature', 'CPI', 'Unemployment', 'Fuel_Price']

In [ ]:
fig8 = make_subplots(rows=2, cols=2, subplot_titles=numerical_features)

for i, col in enumerate(numerical_features):
    row = i // 2 + 1
    col_index = i % 2 + 1

    mean_val = train[col].mean()
    median_val = train[col].median()
    mode_val = train[col].mode().iloc[0]

    # Histogram
    fig8.add_trace(
        go.Histogram(
            x=train[col],
            name=f"{col} Distribution",
            marker=dict(color="#8ECEF5"),
            opacity=0.75,
            nbinsx=40
        ),
        row=row, col=col_index
    )

    # KDE trend line
    x_vals = np.linspace(train[col].min(), train[col].max(), 200)
    kde = gaussian_kde(train[col].dropna())
    y_vals = kde(x_vals)

    # Scale KDE to match histogram height
    y_vals = y_vals * len(train[col]) * ( (train[col].max() - train[col].min()) / 40 )

    fig8.add_trace(
        go.Scatter(
            x=x_vals,
            y=y_vals,
            mode="lines",
            line=dict(color="#41ADF0", width=2),
            name=f"{col} Trend"
        ),
        row=row, col=col_index
    )

    # Mean, Median, Mode lines
    fig8.add_vline(x=mean_val, line=dict(color="#C74846", dash="dash"), row=row, col=col_index)
    fig8.add_vline(x=median_val, line=dict(color="#1D8245", dash="dash"), row=row, col=col_index)
    fig8.add_vline(x=mode_val, line=dict(color="#24579E", dash="dash"), row=row, col=col_index)

fig8.update_layout(
    title="Numerical Feature Distributions",
    title_x=0.5,
    height=800,
    bargap=0.05,
    showlegend=False,
    autosize=True,
    margin=dict(
        l=80, r=80, b=80, t=100, pad=4
    )
)

for i in fig8['layout']['annotations']:
    i['y'] = i['y'] + 0.01

div8 = pio.to_html(fig8, full_html=False, include_plotlyjs=False)
fig8.show()

####Correlation Heatmap of Numerical Features

In [ ]:
numeric_cols = ['Weekly_Sales','Temperature','Fuel_Price','MarkDown1','MarkDown2',
                'MarkDown3','MarkDown4','MarkDown5','CPI','Unemployment','Size']

In [ ]:
corr_matrix = train[numeric_cols].corr()

fig9 = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='ice'
))

fig9.update_layout(
    title={
        'text': 'Correlation Heatmap of Numerical Features',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    xaxis=dict(tickangle=-90, dtick=1),
    yaxis=dict(tickangle=0, dtick=1)
)

div9 = pio.to_html(fig9, full_html=False, include_plotlyjs=False)
fig9.show()

####Insights:

*   Weekly_Sales shows weak direct correlation; sales are influenced by complex interactions.

*   Markdown features (1–5): moderate correlations → represent different promotion types but occur in similar periods.

*   Fuel_Price: Indirect effect (higher fuel prices reduce discretionary spending)

*   Temperature:

  *   Hotter weeks = more beverage/AC sales.
  *   Colder weeks = more winter apparel/heating sales.


*   Store Size: Larger stores generate more sales, but not proportionally (economies of scale, market saturation).


---

=> Direct correlations with sales are weak → need advanced feature engineering.

=> Non-linear models will outperform simple linear regression.

=> Markdown strategies require time-lag modeling.



####Total Markdown Impact

In [ ]:
train['MarkDown_total'] = train[['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']].sum(axis=1)

In [ ]:
fig10 = px.scatter(train,
                 x='MarkDown_total', y='Weekly_Sales',
                 color='Type', trendline='ols',
                 color_continuous_scale='Aggrnyl',
                 title='Total Markdown vs Weekly Sales')

fig10.update_layout(
    title={
        'text': 'Total Markdown vs Weekly Sales',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    yaxis_title='Weekly Sales ($)'
)

div10 = pio.to_html(fig10, full_html=False, include_plotlyjs=False)
fig10.show()

####Insights:

*   Markdown events (discounts/promotions) tend to increase sales, but the effect is not strong.

*   High variance: Many high sales occur even at low markdowns.

*   Diminishing returns: Beyond certain markdown levels (>50k), incremental sales impact plateaus. Deep markdowns don’t guarantee proportional sales boosts.

---

=> Markdowns are useful but not the primary sales driver.

=> Need to optimize markdown strategy → focus on timing (holidays/events) and product targeting, not just markdown size.



####Markdown Impact Distribution by Store Type

In [ ]:
train['MarkDown_total'] = train[['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']].sum(axis=1)
train['Has_MarkDown'] = (train['MarkDown_total'] > 0).astype(int)

In [ ]:
fig11 = px.violin(train,
                y='Weekly_Sales', x='Type',
                color='Has_MarkDown', box=True,
                points='all', title='Sales Distribution: Markdown Impact by Store Type',
                color_discrete_map={0: '#96b2f2', 1: '#ffb2ce'})

fig11.update_layout(
    title={
        'text': 'Sales Distribution: Markdown Impact by Store Type',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    yaxis_title="Weekly Sales ($)"
)

div11 = pio.to_html(fig11, full_html=False, include_plotlyjs=False)
fig11.show()

####Insights:

*   Type 0 & 1 stores: markdowns do not drastically increase weekly sales: sales distributions overlap heavily → markdowns are not significantly driving incremental revenue here (help in clearning inventory)

*   Type 2 stores: markdowns show little to no impact. Weekly sales remain low regardless. (suggesting customer base or demand issues.)


---

=> Markdown is not universally effective.

=> Growth Strategy:

*   Use markdowns selectively in larger/stronger stores (Types 0 & 1).
*   Avoid markdown-heavy strategy in weak stores (Type 2): focus on store repositioning, product mix optimization, or customer acquisition strategies.
*   Invest in demand drivers (marketing, product curation) rather than price cuts





####Sales vs Store Size by Store Type

In [ ]:
fig12 = px.scatter(train,
                 x='Size', y='Weekly_Sales',
                 color='Type', opacity=0.5,
                 color_continuous_scale='Aggrnyl',
                 title='Weekly Sales vs. Store Size',
                 hover_data=['Store', 'Dept'])

fig12.update_layout(
    title={
        'text': 'Weekly Sales vs. Store Size',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    yaxis_title="Weekly Sales ($)"
)

div12 = pio.to_html(fig12, full_html=False, include_plotlyjs=False)
fig12.show()

####Insights:

*   Larger stores (≥150k sq. ft.): higher weekly sales potential.

*   Smaller stores (<50k): generate consistent sales, but at a lower ceiling.

*   Variance within mid-size stores (75k–150k): Wide spread of sales → indicates operational efficiency, location, or product mix matter as much as physical size.

*   Very large stores (200k+) don’t always guarantee the highest sales.

*   Some medium stores outperform them → suggests customer base saturation or inefficient utilization of space.

---

=> Store expansion beyond a certain size has diminishing returns.

=> Growth Strategy:

*   Invest in medium-to-large stores (~100k–150k) in strong locations.
*   Optimize underperforming large stores with better inventory allocation & marketing.
*   Small stores can be viable in dense urban areas where space is premium.





####Sales Seasonality

Yearly Sales

In [ ]:
train['Date'] = pd.to_datetime(train['Date'], errors='coerce')
train['Year'] = train['Date'].dt.year

sales_year = train.groupby('Year')['Weekly_Sales'].sum().reset_index()
# Convert Year to string to make it categorical
sales_year['Year'] = sales_year['Year'].astype(str)

In [ ]:
fig13 = px.bar(
    sales_year,
    x='Year', y='Weekly_Sales',
    title='Total Sales by Year',
    color_discrete_sequence=['#7b91ff'],
    hover_data={'Weekly_Sales': ':,.0f'}
)

fig13.update_layout(
    title={
        'text': 'Total Sales by Year',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    yaxis_title='Weekly Sales ($)'
)

div13 = pio.to_html(fig13, full_html=False, include_plotlyjs=False)
fig13.show()

####Insights:

*   Sales Trend:

    *   2010 → 2011: Sales grew from ~$2.3B to ~$2.45B (growth ~6–7%).
    *   2011 → 2012: Sales dropped sharply to ~$2.0B (decline ~18–20%).


*   2011: peak year → indicates successful drivers (promotions, economic recovery, consumer demand).

*   2012: significant contraction → external shocks or internal operational issues likely.


---

=> October 2012 Strike: did not materially affect Walmart’s business performance → ⬆ public scrutiny of Walmart’s labor practices.

=> Possible causes:

1.   Macroeconomic slowdown.
2.   Higher competition.
3.   Rise of Online Retail. Walmart focused on expanding its physical footprint.
4.   Reduced consumer spending power (inflation, fuel, CPI pressure).
5.   Fewer promotional events or weaker holiday season strategy.



Quaterly Sales

In [ ]:
train['Quarter'] = train['Date'].dt.to_period('Q')
sales_quarter = train.groupby('Quarter')['Weekly_Sales'].sum().reset_index()
sales_quarter['Quarter'] = sales_quarter['Quarter'].astype(str)

In [ ]:
fig14 = px.bar(
    sales_quarter,
    x='Quarter',
    y='Weekly_Sales',
    title='Total Sales by Quarter',
    color_discrete_sequence=['#7b91ff']
)

fig14.update_layout(
    title_x=0.5,
    xaxis_tickangle=-90
)

div14 = pio.to_html(fig14, full_html=False, include_plotlyjs=False)
fig14.show()

####Insights:

*   Seasonal Patterns:

    *   Q1 (Jan–Mar) → post-holiday slump, reduced consumer spending.
    *   2010, 2011 Q4 (Jul–Sep) → strong back-to-school + early holiday shopping.


*   2010 Q4: highest sales (~$710M) → strong seasonal spike.

*   2011 Q4: high sales again (~$680M) → demand stable but not growing.

*   2012 Q4: major disruption (~$200M) → lost sales during the critical holiday quarter.


---

=> Q4 is typically the **largest revenue driver**

=> Possible causes:

1.   Macroeconomic: high unemployment, weaker consumer confidence.
2.   Competitive pressure: loss of market share to rivals (Target, Amazon, etc.).
3.   Internal: weaker promotions, supply chain issues, pricing misalignment.


---

####Recommendations:

*   Strengthen holiday & seasonal strategies: deeper promotions, exclusive deals, targeted advertising.

*   Reduce volatility by diversifying revenue streams (groceries, essentials, services).

*   Investigate 2012Q4 collapse:

    *  systemic (economic): adapt pricing
    *  internal (execution): fix operational gaps.



Monthly Sales

In [ ]:
train['Month'] = train['Date'].dt.month
monthly_sales = train.groupby('Month')['Weekly_Sales'].sum().reset_index()

In [ ]:
fig15 = px.bar(monthly_sales,
             x='Month', y='Weekly_Sales',
             color_discrete_sequence=['#7b91ff'],
             title='Monthly Sales Seasonality')

fig15.update_layout(
    title={
        'text': 'Monthly Sales Seasonality',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    yaxis_title='Weekly Sales ($)'
)

div15 = pio.to_html(fig15, full_html=False, include_plotlyjs=False)
fig15.show()

####Insights:

*   Seasonal Peaks: April and July → highest sales (~$650M):

    *   April: Tax refund spending, Easter-related retail demand.
    *   July: Summer promotions, back-to-school shopping starting.


*   Seasonal Lows: January and November → lowest sales (~\$330M, ~\$400M):

    *   January: Post-holiday spending crash is evident.
    *   July: Underperformance/possible data suppression due to the 2012 Q4 collapse already seen in quarterly data.


---

=> Opportunity: Strengthen holiday promotions (Nov/Dec) to align with industry norms.

=> Risk: Over-reliance on spring and summer peaks — vulnerable if macroeconomic shifts reduce seasonal demand.

---

####Recommendations:

*   Boost Nov/Dec sales with strong Black Friday, holiday campaigns, and online push.

*   Maximize Apr/Jul peaks through marketing, inventory, and promotions.

*   Reduce Jan slump with New Year clearance & essentials.

*   Stabilize sales year-round via groceries, essentials, and services.



Day of Week Sales

In [ ]:
train['DayOfWeek'] = train['Date'].dt.dayofweek
dow_sales = train.groupby('DayOfWeek')['Weekly_Sales'].sum().reset_index()

In [ ]:
fig16 = px.bar(dow_sales,
             x='DayOfWeek', y='Weekly_Sales',
             color_discrete_sequence=['#c2bbf0'],
             title='Sales by Day of Week')

fig16.update_layout(
    title={
        'text': 'Sales by Day of Week',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    yaxis_title='Weekly Sales ($)'
)

div16 = pio.to_html(fig16, full_html=False, include_plotlyjs=False)
fig16.show()

####Insights:

*    Transactions are aggregated/ reported weekly on Thursdays, not daily.

*    Aligns with paycheck cycles (weekly/bi-weekly) → customers stock up midweek.

*    No weekly variation analysis possible (weekend vs weekday trends cannot be studied).

*    Time-series modeling should treat data as weekly observations

In [ ]:
# Check the days that are actually in the dataset
print("Unique days of week:", train['DayOfWeek'].unique())

# Map DayOfWeek to names
day_mapping = {1: "Monday", 2: "Tuesday", 3: "Wednesday", 4: "Thursday",
               5: "Friday", 6: "Saturday", 7: "Sunday"}
unique_day_names = train['DayOfWeek'].map(day_mapping).unique()
print("Unique day names:", unique_day_names)

print("Day of week value counts:")
print(train['DayOfWeek'].value_counts().sort_index())

Unique days of week: [4]
Unique day names: ['Thursday']
Day of week value counts:
DayOfWeek
4    421570
Name: count, dtype: int64


####Month vs Store Type

In [ ]:
sales_heatmap_data = train.groupby(['Type','Month'])['Weekly_Sales'].mean().reset_index()

In [ ]:
sales_heatmap_pivot = sales_heatmap_data.pivot(index='Type', columns='Month', values='Weekly_Sales')

fig17 = go.Figure(data=go.Heatmap(
    z=sales_heatmap_pivot.values,
    x=sales_heatmap_pivot.columns,
    y=sales_heatmap_pivot.index,
    colorscale='Viridis'
))

fig17.update_layout(
    title_text='Month vs Store Type',
    title_x=0.5
)

div17 = pio.to_html(fig17, full_html=False, include_plotlyjs=False)
fig17.show()

####Insights:

*   Type 0 → highest sales consistently across months

    *   Sales baseline: ~$18k–20k
    *   December peak: ~$24k (holiday surge)
    *   Largest holiday uplift: scalable with promotions and stock allocation.

*   Type 1 → moderate sales

    *   Sales baseline: ~$11k–13k
    *   December peak: ~$15k
    *   Secondary contributors: benefit slightly from seasonality

*   Type 2 → low performers

    *   Sales baseline: ~$9k
    *   No significant December boost (almost flat)
    *   Smaller format, niche focus, or limited assortment


---

=> Resource prioritization: Focus marketing, markdowns, and inventory buildup on Type 0 stores in Q4 → highest ROI.

=> Selective optimization: Type 1 stores may need targeted promotions to lift their weaker seasonal gains.

=> Strategic decision: Type 2 stores may be operationally sustained for coverage but are not growth drivers; evaluate cost vs. contribution.





#####Month vs Holiday

In [ ]:
sales_htmp_data = train.groupby(['IsHoliday_x','Month'])['Weekly_Sales'].mean().reset_index()

In [ ]:
sales_heatmap_pivot = sales_htmp_data.pivot(index='IsHoliday_x', columns='Month', values='Weekly_Sales')

fig18 = go.Figure(data=go.Heatmap(
    z=sales_heatmap_pivot.values,
    x=sales_heatmap_pivot.columns,
    y=sales_heatmap_pivot.index,
    colorscale='Viridis',
    texttemplate='%{text}',
    textfont={"size": 12, "color": "white"},
    hoverongaps=False,
    hovertemplate='Holiday: %{y}<br>Month: %{x}<br>Avg Sales: $%{z:,.0f}<extra></extra>'
))

fig18.update_layout(
    title={
        'text': 'Average Sales: Holiday vs Non-Holiday by Month',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    xaxis_title='Month',
    yaxis_title='Holiday Status',
    yaxis=dict(
        tickmode='array',
        tickvals=[0, 1],
        ticktext=['Non-Holiday', 'Holiday']
    ),
    xaxis=dict(
        tickmode='array',
        tickvals=list(range(1, 13)),
        ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    ),
    height=400,
    margin=dict(l=80, r=40, t=80, b=100, pad=4),
    autosize=True
)

div18 = pio.to_html(fig18, full_html=False, include_plotlyjs=False)
fig18.show()

####Insights:

*   Holiday sales spike: November–December → exceeding $22k average

*   Non-holiday sales: December peak (~19k) → around $14k–17k

*   Majority Sales: Q4 holidays → Thanksgiving, Black Friday, Christmas.


---

=> Inventory planning: Stock heavily before Nov–Dec holidays.

=> Promotional leverage: Promotions during holidays yield the highest ROI, especially in Q4.

=> Demand smoothing: Consider smaller promos during off-peak months (Mar–Aug) to reduce sales troughs.

=> Operational readiness: Allocate staff and logistics capacity ahead of holiday months to handle volume surges.



####Store Performance Distribution

In [ ]:
fig19 = px.box(train,
             x='Type', y='Weekly_Sales',
             title='Sales Distribution by Store Type',
             color='Type', color_discrete_sequence=['#FFA8A8', '#B2CFFF', '#AAFFCE'])

fig19.update_layout(
    title={
        'text': 'Sales Distribution by Store Type',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    yaxis_title='Weekly Sales ($)'
)

div19 = pio.to_html(fig19, full_html=False, include_plotlyjs=False)
fig19.show()

####Insights:

*   Type 0:

    *  Moderate weekly sales → max around $480k
    *  More stable → contributes significantly

*   Type 1:

    *  Highest sales distribution → extreme outliers (up to $700k weekly)
    *  Greater variability → strongest revenue driver

*   Type 2:

    *  Lower sales concentration → peaks only around $100k
    *  Narrow distribution → low-performing stores


---

=> Type 0: stability → maintain efficiency and optimize margins.

=> Type 1: growth engine → invest in scaling, inventory, and marketing for these stores.

=> Type 2: underperforms → either reposition (smaller-format/local stores) or consider cost optimization/exit strategy.

=> Outliers in Type 1 & 0 suggest some stores are outperforming peers → best practices from those can be replicated chain-wide.


####Department Performance by Store Type

In [ ]:
dept_type_sales = train.groupby(['Dept', 'Type'])['Weekly_Sales'].mean().reset_index()
top_depts = train.groupby('Dept')['Weekly_Sales'].mean().nlargest(10).index

In [ ]:
fig20 = px.bar(dept_type_sales[dept_type_sales['Dept'].isin(top_depts)],
             x='Dept', y='Weekly_Sales', color='Type',
             color_continuous_scale='viridis',
             title='Top 10 Departments Performance by Store Type')

fig20.update_layout(
    title={
        'text': 'Top 10 Departments Performance by Store Type',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    yaxis_title='Weekly Sales ($)'
)

div20 = pio.to_html(fig20, full_html=False, include_plotlyjs=False)
fig20.show()

####Insights:

*   Dept 40, 90–95:

    *  Very high weekly sales (>150k–200k)
    *  Core revenue drivers: essentials or popular product categories

*   Type 2: Often dominate in top-performing departments, especially around Dept 40 and 92+.

*   Type 1: Strong but slightly lower contributions.

*  Type 0: Participate across most departments but generally at lower sales levels.


---

=> Type 2: star departments → could act as specialized high-margin outlets.

=> Type 1: balanced strength → strong potential for scaling across more departments.

=> Type 0: under-indexes → likely general-purpose but less effective in driving large volumes.

=> Prioritize inventory allocation, pricing, and promotions in departments ~40 and 90–95, with store-type–specific tactics

*  Type 2 = premium push
*  Type 1 = mass scale
*  Type 0 = complementary
